In [6]:
!pip install transformers evaluate datasets
!pip install accelerate -U

In [7]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU Available: True
GPU: Tesla T4


In [8]:
dataset = load_dataset("databricks/databricks-dolly-15k")

dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [11]:
dataset["train"][0]

{'instruction': 'When did Virgin Australia start operating?',
 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.',
 'category': 'closed_qa'}

In [12]:
dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 1502
    })
})

In [13]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [15]:
tokenizer.pad_token = tokenizer.eos_token

In [16]:
def format_example(example):

    prompt = f"""### Question:
{example['instruction']}

### Context:
{example['context']}

### Answer:
{example['response']}"""

    return {"text": prompt}

In [17]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/13509 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [21]:
dataset["train"][0]

{'instruction': 'Given the reference text below the eruption of Mount Vesuvius, how high was the mountain after the disaster?',
 'context': 'In December 1631, Mount Vesuvius in Italy erupted. The eruption began on 16 December 1631 and culminated the day after. The Volcanic Explosivity Index was VEI-5, and it was a Plinian eruption that buried many villages under the resulting lava flows. It is estimated that between 4,000 people were killed by the eruption, making it the highest death toll for a volcanic disaster in the Mediterranean in the last 1800 years.[citation needed] The 1631 eruption was considered to be of minor proportions regarding its eruptive magnitude and erupted volumes compared to the AD 79 eruption, but the damage was not.[citation needed] By the 1631 eruption, the summit of Mount Vesuvius had been reduced by 450m, making its total height lower than that of Mount Somma.',
 'response': 'Mount Vesuvius had a reduced summit by 450 meters.',
 'category': 'closed_qa',
 'tex

In [41]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

In [42]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 1502
    })
})

In [43]:
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/13509 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [44]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1502
    })
})

In [45]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [46]:
model.config.pad_token_id = tokenizer.pad_token_id

model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [47]:
model.config.pad_token_id = tokenizer.pad_token_id

In [48]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [49]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-ml-tutor",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

In [50]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [51]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.604310,2.475580
2,2.387127,2.450757
3,2.223462,2.450631


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=10134, training_loss=2.420322772805828, metrics={'train_runtime': 1573.1313, 'train_samples_per_second': 25.762, 'train_steps_per_second': 6.442, 'total_flos': 5294691090432000.0, 'train_loss': 2.420322772805828, 'epoch': 3.0})

In [52]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
2.223462,2.450631,3


{'eval_loss': 2.4506309032440186}

In [53]:
import math

eval_results = trainer.evaluate()

perplexity = math.exp(eval_results["eval_loss"])

print(f"Perplexity : {perplexity:.2f}")

Training Loss,Validation Loss,Epoch
2.223462,2.450631,3


Perplexity : 11.60


In [54]:
SAVE_PATH = "./gpt2-ml-tutor"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!
